## blend 2 style images

In [ ]:
import os
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications import vgg19

# Generated image size
RESIZE_HEIGHT = 607

NUM_ITER = 1000

# Weights of the different loss components

# If style weight is bigger, output will look more like "Starry Night" than the Paris photo.
CONTENT_WEIGHT = 8e-4
STYLE_WEIGHT = 8e-1

# The layer to use for the content loss.
CONTENT_LAYER_NAME = "block5_conv2"

# List of layers to use for the style loss.
STYLE_LAYER_NAMES = [
    "block1_conv1",
    "block2_conv1",
    "block3_conv1",
    "block4_conv1",
    "block5_conv1",
]


In [ ]:
def get_result_image_size(image_path, result_height):
    image_width, image_height = keras.preprocessing.image.load_img(image_path).size
    result_width = int(image_width * result_height / image_height)
    return result_height, result_width

def preprocess_image(image_path, target_height, target_width):
    img = keras.preprocessing.image.load_img(image_path, target_size=(target_height, target_width))
    arr = keras.preprocessing.image.img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    arr = vgg19.preprocess_input(arr)
    return tf.convert_to_tensor(arr)

In [ ]:
def get_model():
    model = vgg19.VGG19(weights='imagenet', include_top=False)
    outputs_dict = dict([(layer.name, layer.output) for layer in model.layers])
    return keras.Model(inputs=model.inputs, outputs=outputs_dict)

def get_optimizer():
    return keras.optimizers.Adam(
        keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=8.0, decay_steps=445, decay_rate=0.98
        )
    )


In [ ]:
def compute_loss(feature_extractor, combination_image, content_features, style1_features, style2_features, alpha):
    combination_features = feature_extractor(combination_image)
    loss_content = compute_content_loss(content_features, combination_features)
    loss_style = compute_style_loss(style1_features, style2_features, combination_features, combination_image.shape[1] * combination_image.shape[2], alpha)
    return CONTENT_WEIGHT * loss_content + STYLE_WEIGHT * loss_style


def compute_content_loss(content_features, combination_features):
    original_image = content_features[CONTENT_LAYER_NAME]
    generated_image = combination_features[CONTENT_LAYER_NAME]
    return tf.reduce_sum(tf.square(generated_image - original_image)) / 2

def compute_style_loss(style1_features, style2_features, combination_features, combination_size, alpha):
    loss_style = 0
    for layer_name in STYLE_LAYER_NAMES:
        S1 = gram_matrix(style1_features[layer_name][0])
        S2 = gram_matrix(style2_features[layer_name][0])
        S_blended = (1 - alpha) * S1 + alpha * S2

        C = gram_matrix(combination_features[layer_name][0])
        channels = S1.shape[0]

        loss_layer = tf.reduce_sum(tf.square(S_blended - C)) / (4.0 * (channels ** 2) * (combination_size ** 2))
        loss_style += loss_layer / len(STYLE_LAYER_NAMES)
    return loss_style


def style_loss(style_features, combination_features, combination_size):
    S = gram_matrix(style_features)
    C = gram_matrix(combination_features)
    channels = style_features.shape[2]
    return tf.reduce_sum(tf.square(S - C)) / (4.0 * (channels ** 2) * (combination_size ** 2))

def gram_matrix(x):
    x = tf.transpose(x, (2, 0, 1))
    features = tf.reshape(x, (tf.shape(x)[0], -1))
    gram = tf.matmul(features, tf.transpose(features))
    return gram

In [ ]:
def save_result(generated_image, result_height, result_width, name):
    img = deprocess_image(generated_image, result_height, result_width)
    keras.preprocessing.image.save_img(name, img)

def deprocess_image(tensor, result_height, result_width):
    tensor = tensor.numpy()
    tensor = tensor.reshape((result_height, result_width, 3))

    tensor[:, :, 0] += 103.939
    tensor[:, :, 1] += 116.779
    tensor[:, :, 2] += 123.680

    tensor = tensor[:, :, ::-1]
    return np.clip(tensor, 0, 255).astype("uint8")

In [ ]:


content_image_path = "/content/sample_data/paris.jpg"
style_image_path = "/content/sample_data/vg_still.jpg"
style_image_path_2 = "/content/sample_data/picasso_still.jpg"

result_height, result_width = get_result_image_size(content_image_path, RESIZE_HEIGHT)
print("result resolution: (%d, %d)" % (result_height, result_width))

content_tensor = preprocess_image(content_image_path, result_height, result_width)
style_tensor = preprocess_image(style_image_path, result_height, result_width)
style2_tensor = preprocess_image(style_image_path_2, result_height, result_width)
generated_image = tf.Variable(tf.random.uniform(style_tensor.shape, dtype=tf.dtypes.float32))

alpha = 0.75

model = get_model()
optimizer = get_optimizer()
print(model.summary())

content_features = model(content_tensor)
style1_features = model(style_tensor)
style2_features = model(style2_tensor)

for iter in range(NUM_ITER):
    with tf.GradientTape() as tape:
        loss = compute_loss(model, generated_image, content_features, style1_features, style2_features, alpha)

    grads = tape.gradient(loss, generated_image)
    print("iter: %4d, loss: %8.f" % (iter, loss))
    optimizer.apply_gradients([(grads, generated_image)])

    if (iter + 1) % 100 == 0:
        name = "/content/sample_data/generated_at_iteration_%d.png" % (iter + 1)
        save_result(generated_image, result_height, result_width, name)

final_name = f"/content/sample_data/result_{NUM_ITER}_a{alpha:.2f}_cw{CONTENT_WEIGHT:.1e}_sw{STYLE_WEIGHT:.1e}.png"
save_result(generated_image, result_height, result_width, final_name)




result resolution: (607, 910)
80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None, None, 3)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, None, None, 64) │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, None, None, 64) │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, None, None, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, None, None,     │        73,856 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, None, None,     │       147,584 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, None, None,     │             0 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, None, None,     │       295,168 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, None, None,     │       590,080 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, None, None,     │       590,080 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv4 (Conv2D)           │ (None, None, None,     │       590,080 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, None, None,     │             0 │
│                                 │ 256)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, None, None,     │     1,180,160 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, None, None,     │     2,359,808 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, None, None,     │     2,359,808 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv4 (Conv2D)           │ (None, None, None,     │     2,359,808 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, None, None,     │             0 │
│                                 │ 512)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, None, None,     │     2,359,808 │
│                                 │ 512)                   │             

 Total params: 20,024,384 (76.39 MB)

 Trainable params: 20,024,384 (76.39 MB)

 Non-trainable params: 0 (0.00 B)

None


/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(1, 607, 910, 3))
  warnings.warn(msg)


iter:    0, loss:  2136284
iter:    1, loss:  2006355
iter:    2, loss:  1771206
iter:    3, loss:  1627022
iter:    4, loss:  1542774
iter:    5, loss:  1430388
iter:    6, loss:  1303722
iter:    7, loss:  1183175
iter:    8, loss:  1072207
iter:    9, loss:   977247
iter:   10, loss:   897031
iter:   11, loss:   829239
iter:   12, loss:   772490
iter:   13, loss:   725316
iter:   14, loss:   686140
iter:   15, loss:   650367
iter:   16, loss:   618537
iter:   17, loss:   589708
iter:   18, loss:   562625
iter:   19, loss:   537710
iter:   20, loss:   514336
iter:   21, loss:   491438
iter:   22, loss:   470526
iter:   23, loss:   451483
iter:   24, loss:   433707
iter:   25, loss:   417344
iter:   26, loss:   401638
iter:   27, loss:   386943
iter:   28, loss:   373288
iter:   29, loss:   360407
iter:   30, loss:   348449
iter:   31, loss:   338096
iter:   32, loss:   327733
iter:   33, loss:   318474
iter:   34, loss:   309596
iter:   35, loss:   300877
iter:   36, loss:   292268
i